# CREAM Progressive Graph Noise

Accuracy as a function of graph distance. Noise flips uniformly selected adjacency entries, including diagonal entries.


In [ ]:
from pathlib import Path
import pandas as pd
dataset_roots = {
  "cub": "../experiments/CUB/train_cbm/Standard_CUB/cub_progressive_noise/CREAM_cub_progressive_noise"
}
metadata_paths = {
  "cub": "../data/CUB/progressive_noise_graph/progressive_noise_metadata.csv"
}
rows = []
for dataset, root_str in dataset_roots.items():
    root = Path(root_str)
    meta = pd.read_csv(metadata_paths[dataset])
    for csv_path in sorted(root.glob('noise_*/last_metrics/*.csv')):
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        row = df.iloc[0].to_dict()
        row['dataset'] = dataset
        row['noise_percent'] = int(csv_path.parents[1].name.replace('noise_', ''))
        row['csv_path'] = str(csv_path)
        rows.append(row)
results = pd.DataFrame(rows)
metadata = pd.concat([pd.read_csv(path) for path in metadata_paths.values()], ignore_index=True)
merged = results.merge(metadata, on=['dataset', 'noise_percent'], how='left') if not results.empty else results
cols = [c for c in ['dataset', 'noise_percent', 'graph_distance', 'different_edges', 'added_edges', 'deleted_edges', 'perturbed_edges', 'test_task_accuracy', 'test_concept_accuracy', 'test_dropout_task_accuracy'] if c in merged.columns]
merged[cols].sort_values(['dataset', 'noise_percent']) if not merged.empty else merged


In [ ]:
import matplotlib.pyplot as plt
if not merged.empty and 'test_task_accuracy' in merged.columns:
    for dataset, df in merged.groupby('dataset'):
        df = df.sort_values('graph_distance')
        ax = df.plot(x='graph_distance', y='test_task_accuracy', marker='o', figsize=(7, 4), legend=False)
        ax.set_xlabel('Graph distance = different edge positions / total positions')
        ax.set_ylabel('Test task accuracy')
        ax.set_title(f'{dataset}: accuracy vs progressive graph noise')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
